In [1]:
from aiida import orm, load_profile

load_profile()

Profile<uuid='1a5a8d0836814a04a238c67cc7481655' name='default'>

In [301]:
import numpy as np

from pydantic import BaseModel, Field, ConfigDict, field_validator, computed_field, create_model, model_validator
from typing import Optional, List, Dict, Any, Union


class PropertiesModel(BaseModel):
    """
    A model to represent properties of a material.
    """
    
    pbc: list[bool] = Field(
        default=[True, True, True],
        description="Periodic boundary conditions",
        store_in_db=True,
        min_items=3,
        max_items=3,
        target="global",
    )
    
    
    cell: Union[np.ndarray[float], list[float]] = Field(
        default=None,
        description="Lattice vectors",
        units="Angstrom",
        store_in_db=False,
        target="global",
    )
    
    symbols: list[str] = Field(
        default=None,
        description="Chemical symbols of the atoms",
        store_in_db=False,
        target="sites-global",
    )
    
    kinds: dict[str, list[int]] = Field(
        default=None,
        description="Mapping of chemical symbols to indices",
        store_in_db=False,
        target="sites-compression",
    )
    
    positions: Union[np.ndarray[float], list[float]] = Field(
        default=None,
        description="3D coordinates of the atoms",
        units="Angstrom",
        store_in_db=False,
        target="sites-global",
    )
    
    magmom: Union[np.ndarray[float], list[float]] = Field(
        default=None,
        description="3D magnetic moment vector per site",
        units="Bohr magneton",
        store_in_db=False,
        target="sites",
    )
    
    charge: Union[np.ndarray[float], list[float]] = Field(
        default=0,
        description="Charge of the atoms",
        units="e",
        store_in_db=False,
        target="sites",
    )
    
    model_config = ConfigDict(arbitrary_types_allowed=True)
    
    @field_validator('positions', mode='after')
    @classmethod
    def validate_positions(cls, v: int) -> int:
        num_sites = np.shape(v)[0]
        dimensions = np.shape(v)[1]

        if num_sites == 0:
            raise ValueError("The number of sites must be greater than 0.")
        
        if dimensions != 3:
            raise ValueError("The dimensions of the positions must be 3.")
        
        # TODO: validate that the positions are not overlapping
        
        return v
    
    @model_validator(mode='after')
    @classmethod
    def validate_model(cls, values: Dict[str, Any]) -> Dict[str, Any]:
        """
        Validate the model after all required fields have been validated.
        """

        num_sites = np.shape(values.positions)[0]
        #num_symbols = len(values.symbols)
        
        if values.kinds is not None:
            use_kinds = True
            expected_len = len(values.kinds.keys())
        else:
            use_kinds = False
            expected_len = num_sites
        
        # Loop through the fields
        for field_name, field_info in values.model_fields.items():
            print(f"Field Name: {field_name}")
            print(f"Field Description: {field_info.description}")
            print(f"Field Value: {getattr(values, field_name)}")
            print("-" * 40)
            
        if values.magmom is not None:
            magmom = np.array(values.magmom)
            if magmom.shape[0] != expected_len:
                raise ValueError(f"The number of input magnetic moments should be {expected_len}, i.e. should match the number of sites (or kinds, if defined).")
            if magmom.shape[1] != 3:
                raise ValueError("The dimensions of the magnetic moments must be 3.")

            if use_kinds:
                magmom = np.zeros((num_sites, 3))
                for kind, indices in values.kinds.items():
                    magmom[indices] = values.magmom[indices]
                values.magmom = magmom
            
        return values
        
    @computed_field
    def has_magmom(self) -> bool:
        """
        Computed field.
        """
        if self.magmom is None:
            return False
        return  len(self.magmom) > 0
        
    def save_property_to_npy(self, property_name: str, filename: str) -> None:
        """
        Save the properties to a .npy file.
        """
        np.save(filename, self.model_dump()[property_name])
    
    def save_compressed_property_to_npy(self, filename: str, compress = False) -> None:
        """
        Save the properties to a compressed .npz file.
        """

        if self.model_dump().get("kinds", None):
            compress = True
        elif compress:
            raise ValueError("The property 'kinds' is not defined. Cannot save compressed file.")

        if not compress:
            properties = self.model_dump()
        else:
            # self.validate_kinds()
            raise NotImplementedError("Compressed saving is not implemented yet.")

        sparse_properties = {}

        for key, value in properties.items():
            # Find indices where charge > 0
            print(key, value)
            if key in self.model_fields.keys():
                if self.model_fields[key].json_schema_extra["target"] == "sites" and not self.model_fields[key].json_schema_extra["store_in_db"]:
                    mask = np.any(value != self.model_fields[key].default, axis=1 if len(value.shape) > 1 else 0)
                    indices = np.where(mask)[0].astype(int)
                    # Create a new array with indices in first column and values in second
                    sparse_properties[key] = np.column_stack((indices, value[indices]))

        np.savez_compressed(filename, **sparse_properties)

In [302]:
p = PropertiesModel(
    symbols=["Fe", "O"],
    positions=np.array([[1,2,3],[2,1,3]]),
    magmom=np.array([[1.0, 0.0, 0.0],[-1.0, 0.0, 0.0]]),
    charge=np.array([0.0, 1]),
    )

Field Name: pbc
Field Description: Periodic boundary conditions
Field Value: [True, True, True]
----------------------------------------
Field Name: cell
Field Description: Lattice vectors
Field Value: None
----------------------------------------
Field Name: symbols
Field Description: Chemical symbols of the atoms
Field Value: ['Fe', 'O']
----------------------------------------
Field Name: kinds
Field Description: Mapping of chemical symbols to indices
Field Value: None
----------------------------------------
Field Name: positions
Field Description: 3D coordinates of the atoms
Field Value: [[1 2 3]
 [2 1 3]]
----------------------------------------
Field Name: magmom
Field Description: 3D magnetic moment vector per site
Field Value: [[ 1.  0.  0.]
 [-1.  0.  0.]]
----------------------------------------
Field Name: charge
Field Description: Charge of the atoms
Field Value: [0. 1.]
----------------------------------------


In [303]:
p.magmom

array([[ 1.,  0.,  0.],
       [-1.,  0.,  0.]])

In [304]:
p.model_dump()

{'pbc': [True, True, True],
 'cell': None,
 'symbols': ['Fe', 'O'],
 'kinds': None,
 'positions': array([[1, 2, 3],
        [2, 1, 3]]),
 'magmom': array([[ 1.,  0.,  0.],
        [-1.,  0.,  0.]]),
 'charge': array([0., 1.]),
 'has_magmom': True}

In [305]:
#p.save_property_to_npy("magmom", "/home/aiida/work/SDATA/magmom.npy")

In [306]:
p.save_compressed_property_to_npy("/home/aiida/work/SDATA/properties.npz")

pbc [True, True, True]
cell None
symbols ['Fe', 'O']
kinds None
positions [[1 2 3]
 [2 1 3]]
magmom [[ 1.  0.  0.]
 [-1.  0.  0.]]
charge [0. 1.]
has_magmom True


/tmp/ipykernel_712756/1730117229.py:164: DeprecationWarning: Calling nonzero on 0d arrays is deprecated, as it behaves surprisingly. Use `atleast_1d(cond).nonzero()` if the old behavior was intended. If the context of this warning is of the form `arr[nonzero(cond)]`, just use `arr[cond]`.
  indices = np.where(mask)[0].astype(int)


In [307]:
#np.load("/home/aiida/work/SDATA/properties.npz")

In [308]:
with np.load("/home/aiida/work/SDATA/properties.npz") as data:
    for k, v in data.items():
        print(k, v)

magmom [[ 0.  1.  0.  0.]
 [ 1. -1.  0.  0.]]
charge [[0. 0.]]


In [267]:
np.where(p.magmom != p.model_fields["magmom"].default)

(array([0, 0, 0, 1, 1, 1]), array([0, 1, 2, 0, 1, 2]))

In [268]:
print(p.schema_json(indent=2))

{
  "description": "A model to represent properties of a material.",
  "properties": {
    "pbc": {
      "default": [
        true,
        true,
        true
      ],
      "description": "Periodic boundary conditions",
      "items": {
        "type": "boolean"
      },
      "maxItems": 3,
      "minItems": 3,
      "store_in_db": true,
      "target": "global",
      "title": "Pbc",
      "type": "array"
    },
    "cell": {
      "default": null,
      "description": "Lattice vectors",
      "items": {
        "type": "number"
      },
      "store_in_db": false,
      "target": "global",
      "title": "Cell",
      "type": "array",
      "units": "Angstrom"
    },
    "symbols": {
      "default": null,
      "description": "Chemical symbols of the atoms",
      "items": {
        "type": "string"
      },
      "store_in_db": false,
      "target": "sites-global",
      "title": "Symbols",
      "type": "array"
    },
    "kinds": {
      "additionalProperties": {
        "ite

In [40]:
p.has_magmom

True

In [27]:
p = PropertiesModel(
    positions=np.array([[1,2,3],[1,2,3]]),
    kinds={"Fe1": [0], "Fe2": [1]},
    )

Field Name: pbc
Field Description: Periodic boundary conditions
Field Value: [True, True, True]
----------------------------------------
Field Name: cell
Field Description: Lattice vectors
Field Value: None
----------------------------------------
Field Name: symbols
Field Description: Chemical symbols of the atoms
Field Value: None
----------------------------------------
Field Name: kinds
Field Description: Mapping of chemical symbols to indices
Field Value: {'Fe1': [0], 'Fe2': [1]}
----------------------------------------
Field Name: positions
Field Description: 3D coordinates of the atoms
Field Value: [[1 2 3]
 [1 2 3]]
----------------------------------------
Field Name: magmom
Field Description: 3D magnetic moment vector per site
Field Value: None
----------------------------------------
Field Name: charge
Field Description: Charge of the atoms
Field Value: 0
----------------------------------------


In [30]:
positions=np.array([[1,2,3],[1,2,3],[3,4,5]])
kinds={
    "Fe1": {
        "site_indices": [0,2],
        "magmom": np.array([[1.0, 0.0, 0.0]])
        }, 
    "Fe2": {
        "site_indices": [1],
        },
}

In [95]:
# For 1D arrays like your charge data
charge = np.array([[0.0, 1.0, 0.0], [0.0, 1.0, 0.0], [0.0, 0.0, 0.0], [0.0, 1.0, 0.0]])
# Find indices where charge > 0
indices = np.where(charge != [0.0, 0.0, 0.0])[0]
# Create a new array with indices in first column and values in second
result = np.column_stack((indices, charge[indices]))
print(result)
# Output: [[1 1.0]]  # Index 1 has value 1.0

[[0. 0. 1. 0.]
 [1. 0. 1. 0.]
 [3. 0. 1. 0.]]


In [96]:
np.where(charge != [0.0, 0.0, 0.0])

(array([0, 1, 3]), array([1, 1, 1]))

In [97]:
result[:,0]

array([0., 1., 3.])